## 1. Setup & Environment Configuration

In [ ]:
!pip install langchain langchain-openai langsmith langchain-community -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening-system"
os.environ["LANGCHAIN_API_KEY"] = "your_api_key"  
os.environ["OPENAI_API_KEY"] = "your_openai_api_key"

print("✅ Environment configured.")
print(f"   LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")
print(f"   Tracing Enabled: {os.environ['LANGCHAIN_TRACING_V2']}")

✅ Environment configured.
   LangSmith Project: resume-screening-system
   Tracing Enabled: true


## 2. LLM Initialization

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,   
)

print(f"✅ LLM initialized: {llm.model_name}")

C:\Users\Joshith\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM initialized: gpt-4o-mini


## 3. Prompt Templates

In [5]:
from langchain_core.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert resume parser. Extract information ONLY from the resume provided.
Do NOT assume, infer, or hallucinate any skills not explicitly mentioned.

Resume:
{resume}

Extract:
1. Skills: (list all technical and soft skills)
2. Experience: (years and domains)
3. Tools & Technologies: (frameworks, libraries, platforms)

Rules:
- Only extract what is explicitly stated.
- If missing, write 'Not mentioned'.

Output:
"""
)

matching_prompt = PromptTemplate(
    input_variables=["job_description", "extracted_profile"],
    template="""
You are a technical recruiter comparing a candidate profile against a job description.

Job Description:
{job_description}

Candidate Profile:
{extracted_profile}

Perform a structured comparison:
1. Matched Skills: (skills present in both)
2. Missing Skills: (required in JD but absent in resume)
3. Bonus Skills: (extra skills beyond JD requirements)
4. Experience Match: (does experience level satisfy JD?)

Rules:
- Be objective. Do NOT give benefit of the doubt for missing skills.

Output:
"""
)

scoring_prompt = PromptTemplate(
    input_variables=["match_analysis"],
    template="""
You are an AI hiring evaluator. Based on the match analysis, assign a fit score.

Match Analysis:
{match_analysis}

Scoring Rubric:
- Skills coverage: 50 points max
- Experience relevance: 30 points max
- Bonus/additional skills: 20 points max

Rules:
- Score must be between 0 and 100.
- Deduct points fairly for missing required skills.
- Do NOT inflate scores.

Respond with:
Score: <number>
Breakdown: <brief breakdown>

Output:
"""
)

explanation_prompt = PromptTemplate(
    input_variables=["score", "match_analysis", "job_description"],
    template="""
You are a senior recruiter writing a hiring recommendation.

Candidate Fit Score: {score}
Match Analysis: {match_analysis}
Job Description Summary: {job_description}

Write a clear, honest explanation (3-5 sentences) covering:
1. Why the candidate received this score
2. Key strengths relevant to the role
3. Critical gaps or concerns
4. Final recommendation (Strong Hire / Maybe / No Hire)

Rules:
- Be professional and factual.
- Ground every claim in the match analysis.

Explanation:
"""
)

print("✅ All 4 prompt templates defined.")

✅ All 4 prompt templates defined.


## 4. Build LCEL Chains

In [6]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

extraction_chain  = extraction_prompt  | llm | parser
matching_chain    = matching_prompt    | llm | parser
scoring_chain     = scoring_prompt     | llm | parser
explanation_chain = explanation_prompt | llm | parser

print("✅ LCEL chains built:")
print("   extraction_chain  → matching_chain → scoring_chain → explanation_chain")

✅ LCEL chains built:
   extraction_chain  → matching_chain → scoring_chain → explanation_chain


## 5. Sample Data

In [7]:
JOB_DESCRIPTION = """
Position: Data Scientist | Company: TechVision Analytics

Requirements:
- 2+ years experience in data science or ML
- Python (NumPy, Pandas, Scikit-learn)
- Deep learning frameworks (TensorFlow or PyTorch)
- Strong SQL and database querying
- Data visualization (Matplotlib, Seaborn, or Tableau)
- MLOps tools (MLflow, Docker, or Kubernetes)
- ML model deployment via REST APIs (Flask/FastAPI)
- Statistical methods and ML algorithms

Nice to have: NLP/CV experience, Cloud (AWS/GCP/Azure), A/B testing
"""

RESUME_STRONG = """
Name: Ananya Krishnan
EXPERIENCE:
Senior Data Scientist – DataCore Solutions (2022–2025)
- NLP classifier using BERT (PyTorch), 94% accuracy
- Deployed ML via FastAPI on AWS EC2, 10,000+ daily requests
- MLflow for experiment tracking, Docker for containerization
- Performed A/B testing in production
Data Analyst – InfoSys Ltd. (2021–2022)
- Complex SQL on PostgreSQL; Tableau dashboards
SKILLS: Python, PyTorch, TensorFlow, Scikit-learn, NumPy, Pandas, SQL,
FastAPI, Docker, MLflow, AWS, NLP, Transformers, Matplotlib, Seaborn
EDUCATION: B.E. CS – BITS Pilani (2021) | CGPA: 9.1
"""

RESUME_AVERAGE = """
Name: Rahul Mehta
EXPERIENCE:
Data Analyst – RetailMart Inc. (2023–2025)
- Analyzed sales data using Python/Pandas, SQL for segmentation
- Built Logistic Regression, Decision Tree models (Scikit-learn)
- Matplotlib visualizations
PROJECTS: House Price Prediction (Linear Regression), Customer Churn (Random Forest)
SKILLS: Python, Pandas, NumPy, Scikit-learn, SQL, Matplotlib, Git
EDUCATION: B.Sc. Statistics – Univ. of Mumbai (2022) | CGPA: 7.8
"""

RESUME_WEAK = """
Name: Priya Sharma
EXPERIENCE: None (fresher)
PROJECTS: Titanic EDA (tutorial-based), Excel Sales Dashboard
SKILLS: Python (beginner), Excel, Basic statistics
EDUCATION: B.Com – Bangalore University (2024) | CGPA: 6.5
"""

RESUMES = [
    ("Strong – Ananya Krishnan", RESUME_STRONG),
    ("Average – Rahul Mehta", RESUME_AVERAGE),
    ("Weak – Priya Sharma", RESUME_WEAK),
]

print("✅ Sample data loaded: 3 resumes + 1 job description.")

✅ Sample data loaded: 3 resumes + 1 job description.


## 6. Run Screening Pipeline

In [ ]:
all_results = []

for label, resume in RESUMES:
    print(f"\n{'='*55}")
    print(f"  Processing: {label}")
    print(f"{'='*55}")

    print("[1/4] Extracting skills...")
    extracted = extraction_chain.invoke({"resume": resume})

    print("[2/4] Matching against JD...")
    match = matching_chain.invoke({
        "job_description": JOB_DESCRIPTION,
        "extracted_profile": extracted
    })

    print("[3/4] Scoring candidate...")
    score = scoring_chain.invoke({"match_analysis": match})

    print("[4/4] Generating explanation...")
    explanation = explanation_chain.invoke({
        "score": score,
        "match_analysis": match,
        "job_description": JOB_DESCRIPTION
    })

    result = {
        "candidate": label,
        "extracted_profile": extracted,
        "match_analysis": match,
        "score_output": score,
        "explanation": explanation
    }
    all_results.append(result)

    print(f"\n✅ SCORE:\n{score}")
    print(f"\n💡 EXPLANATION:\n{explanation}")

print("\n🏁 All 3 candidates screened!")